# Lab 4: Machine Translation with a Pre-trained Transformer

**Task:** Translate English → Romanian with a pre-trained sequence-to-sequence Transformer (T5), then evaluate the translations with BLEU. An optional final section translates the other way (Romanian → English), as the lab title suggests.

**Reference notebook:** Surdeanu & Valenzuela-Escárcega, *A Gentle Introduction to Deep Learning for Natural Language Processing*, `chap15_translation_en_to_ro.ipynb` (https://github.com/clulab/gentlenlp).

| Section | Contents |
|---|---|
| Part A | How machine translation from English to a foreign language works (< 500 words) |
| Part B | Implementation: (1) setup, (2) tokenizer + model, (3) dataset, (4) translate method, (5) BLEU |
| Part C | Analysis: what each command does, and how to read the BLEU results |
| Extension | Romanian → English with a MarianMT model |
| References | Sources cited |

> **Runtime:** a GPU (in Colab: *Runtime → Change runtime type → GPU*) translates the full test set (1,999 sentences) in about a minute. On a CPU, set `SAMPLE_SIZE` in Part B.3 to a few hundred sentences.

---
# Part A: How Machine Translation from English to a Foreign Language Works

**Machine translation (MT)** maps a sentence in a source language (English) to a sentence with the same meaning in a target language (here, Romanian). Modern **neural MT** treats translation as conditional text generation. The model learns $P(y \mid x)$ and generates the target one token at a time, with each token conditioned on the source and on the tokens produced so far.

**Encoder–decoder architecture.** Our model, T5 (Raffel et al., 2020), is a Transformer encoder–decoder (Vaswani et al., 2017).
- The **encoder** reads the whole English sentence. Its self-attention builds a contextual vector for every token, so "bank" in "river bank" is represented differently from "bank" in "bank account".
- The **decoder** generates Romanian tokens left to right. *Masked self-attention* lets it see only earlier outputs. *Cross-attention* lets it look at every encoder vector, which is how it learns soft word alignments.

**Subword tokenization.** Text is split into SentencePiece subwords (Kudo & Richardson, 2018). This keeps the vocabulary small and avoids unknown words. That matters for Romanian, which is highly inflected and uses diacritics (ă, â, î, ș, ț).

**Pre-training and task prefixes.** T5 frames every task as text-to-text. It was pre-trained on web text and on a multi-task mixture that includes WMT English–Romanian translation. A plain-text prefix selects the task: `"translate English to Romanian: The cat sleeps."` should produce `"Pisica doarme."`. We can therefore use the model **without fine-tuning**.

**Decoding.** *Greedy decoding* takes the most probable token at each step. *Beam search* keeps the *k* best partial translations and usually finds better outputs. Generation stops at the end-of-sequence token `</s>` or at a maximum length, and the token IDs are decoded back into text.

**Pipeline in this lab:**
1. Load a pre-trained tokenizer and `t5-small` (≈60M parameters) from the Hugging Face Hub.
2. Load the WMT16 English–Romanian test set (Bojar et al., 2016).
3. Add the prefix to each sentence, tokenize in padded batches, run `model.generate`, and decode.
4. Score the outputs against human references with **BLEU** (Papineni et al., 2002).

**Evaluation with BLEU.** BLEU combines the 1- to 4-gram precisions of the output against the reference with a geometric mean. It multiplies the result by a **brevity penalty** for outputs that are too short. Scores range from 0 to 100. **SacreBLEU** (Post, 2018) standardizes tokenization so that scores can be compared across papers.

**Limitations.** BLEU rewards surface overlap, so a valid paraphrase that differs from the single reference is penalized. Small models give fluent but sometimes literal or wrong translations. Larger models, fine-tuning and better decoding improve quality.

---
# Part B: Model Implementation

## B.1 Set up torch, numpy and transformers

In [ ]:
# Install the libraries (already present on Colab except `evaluate` and `sacrebleu`)
!pip install -q transformers datasets evaluate sacrebleu sentencepiece

In [ ]:
import random
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm

# Reproducibility: fix every random number generator we rely on
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use the GPU when one is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch        {torch.__version__}")
print(f"numpy        {np.__version__}")
print(f"transformers {transformers.__version__}")
print(f"device       {device}")

## B.2 Load the tokenizer and the pre-trained model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"   # ~60M parameters. "t5-base" gives better translations but runs slower.

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)
model.eval()              # inference mode: disables dropout

print(type(tokenizer).__name__, "| vocabulary size:", tokenizer.vocab_size)
print(type(model).__name__, "| parameters:", f"{model.num_parameters():,}")

In [ ]:
# Look at how the tokenizer splits a sentence into subword pieces
example = "translate English to Romanian: The weather is beautiful today."
enc = tokenizer(example)
print("Tokens :", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("IDs    :", enc["input_ids"])

## B.3 Load the dataset from the Hugging Face Hub

In [ ]:
from datasets import load_dataset

# WMT16 English-Romanian. "wmt/wmt16" is the current Hub ID (older code uses just "wmt16").
dataset = load_dataset("wmt/wmt16", "ro-en")
print(dataset)

In [ ]:
test_data = dataset["test"]

# Optional: work on a random subset when running on a CPU (None = full test set of 1,999 pairs)
SAMPLE_SIZE = None
if SAMPLE_SIZE is not None:
    test_data = test_data.shuffle(seed=SEED).select(range(SAMPLE_SIZE))

# Each example looks like {"translation": {"en": "...", "ro": "..."}}
print(test_data[0])
print(f"\nNumber of test pairs: {len(test_data)}")

In [ ]:
# Pull the parallel sentences into plain Python lists
sources    = [ex["translation"]["en"] for ex in test_data]   # English input
references = [ex["translation"]["ro"] for ex in test_data]   # Romanian human translation

lengths = [len(s.split()) for s in sources]
print(f"Average English length: {np.mean(lengths):.1f} words (max {np.max(lengths)})")
for i in range(3):
    print(f"\nEN: {sources[i]}\nRO: {references[i]}")

## B.4 Implement the translate method and apply it to the dataset

In [ ]:
PREFIX = "translate English to Romanian: "   # T5 task prefix used during its pre-training

def translate(sentences, batch_size=32, num_beams=4, max_length=256):
    '''Translate a list of English sentences into Romanian with the loaded T5 model.'''
    translations = []
    for start in tqdm(range(0, len(sentences), batch_size), desc="Translating"):
        batch = [PREFIX + s for s in sentences[start:start + batch_size]]

        # Tokenize: pad to the longest sentence in the batch, truncate overly long inputs
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=max_length).to(device)

        # Generate: beam search, no gradient tracking
        with torch.no_grad():
            output_ids = model.generate(**inputs, num_beams=num_beams,
                                        max_length=max_length, early_stopping=True)

        # Turn token IDs back into text, dropping <pad> and </s>
        translations.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return translations

# Quick check on a single sentence
print(translate(["The cat is sleeping on the sofa."]))

In [ ]:
# Apply the method to the whole test set
predictions = translate(sources)

for i in range(5):
    print(f"EN  : {sources[i]}")
    print(f"REF : {references[i]}")
    print(f"PRED: {predictions[i]}\n")

## B.5 Evaluate translation quality with BLEU

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")

# SacreBLEU expects one list of references per prediction (it supports several references)
results = bleu.compute(predictions=predictions,
                       references=[[r] for r in references])

print(f"BLEU score           : {results['score']:.2f}")
print("n-gram precisions (%):", [round(p, 1) for p in results["precisions"]])
print(f"Brevity penalty      : {results['bp']:.3f}")
print(f"Hyp/ref length ratio : {results['sys_len'] / results['ref_len']:.3f} "
      f"({results['sys_len']} / {results['ref_len']} tokens)")

In [ ]:
# Romanian diacritics come in two Unicode forms: comma-below (ș ț, correct) and
# cedilla (ş ţ, legacy). If the model and the references use different forms,
# BLEU is lowered for reasons that have nothing to do with translation quality.
# Here we map both forms to the same character and score again.
def normalize_ro(text):
    table = str.maketrans({"ş": "ș", "Ş": "Ș", "ţ": "ț", "Ţ": "Ț"})
    return text.translate(table)

results_norm = bleu.compute(predictions=[normalize_ro(p) for p in predictions],
                            references=[[normalize_ro(r)] for r in references])
print(f"BLEU (raw)                   : {results['score']:.2f}")
print(f"BLEU (diacritics normalized) : {results_norm['score']:.2f}")

In [ ]:
# Sentence-level BLEU to find the best and worst translations
import sacrebleu

sent_scores = [sacrebleu.sentence_bleu(p, [r]).score
               for p, r in zip(predictions, references)]
order = np.argsort(sent_scores)

def show(idx):
    print(f"[sentence BLEU = {sent_scores[idx]:.1f}]")
    print(f"EN  : {sources[idx]}\nREF : {references[idx]}\nPRED: {predictions[idx]}\n")

print("=== 3 worst translations ===")
for i in order[:3]:
    show(i)
print("=== 3 best translations ===")
for i in order[-3:]:
    show(i)

In [ ]:
# Does quality fall off for longer sentences?
import matplotlib.pyplot as plt

bins = [0, 10, 20, 30, 40, 1000]
labels = ["1-10", "11-20", "21-30", "31-40", "41+"]
bucket_bleu, bucket_n = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    idx = [i for i, n in enumerate(lengths) if lo < n <= hi]
    bucket_n.append(len(idx))
    bucket_bleu.append(bleu.compute(predictions=[predictions[i] for i in idx],
                                    references=[[references[i]] for i in idx])["score"] if idx else 0.0)

for lab, n, b in zip(labels, bucket_n, bucket_bleu):
    print(f"{lab:>6} words: {n:4d} sentences, BLEU {b:5.2f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(labels, bucket_bleu, color="#4C72B0")
ax.set_xlabel("English source length (words)")
ax.set_ylabel("Corpus BLEU")
ax.set_title("BLEU by source-sentence length (t5-small, EN→RO)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Decoding ablation: greedy search (1 beam) vs. beam search (4 beams) on 200 sentences
sub = list(range(min(200, len(sources))))
greedy = translate([sources[i] for i in sub], num_beams=1)
beam4  = [predictions[i] for i in sub]
refs   = [[references[i]] for i in sub]
print(f"Greedy (1 beam)  BLEU: {bleu.compute(predictions=greedy, references=refs)['score']:.2f}")
print(f"Beam   (4 beams) BLEU: {bleu.compute(predictions=beam4,  references=refs)['score']:.2f}")

In [ ]:
# Automatic summary of the results (the numbers come from the run above)
s, bp = results["score"], results["bp"]
ratio = results["sys_len"] / results["ref_len"]
band = ("low: translations are often wrong or hard to understand" if s < 10 else
        "fair: the gist comes through but with many errors" if s < 20 else
        "good: translations are understandable and often close to the reference" if s < 30 else
        "high: quality approaching strong production systems")
print(f"Corpus BLEU = {s:.2f} -> {band}.")
print(f"Unigram precision {results['precisions'][0]:.1f}% vs 4-gram precision {results['precisions'][3]:.1f}%: "
      "word choice is much better than exact phrase order and morphology.")
print(f"Length ratio {ratio:.3f} (BP = {bp:.3f}): " +
      ("outputs are not shorter than references, so there is no length penalty." if bp >= 0.999
       else "outputs are somewhat short, and the brevity penalty lowers the score."))

---
# Part C: Analysis

## C.1 How the commands in Part B work

**B.1 Setup**
- `pip install transformers datasets evaluate sacrebleu sentencepiece` installs the required packages:
  - `transformers` provides the models and tokenizers.
  - `datasets` downloads and caches corpora from the Hub.
  - `evaluate` and `sacrebleu` compute BLEU.
  - `sentencepiece` is the subword library behind the T5 tokenizer.
- `random.seed`, `np.random.seed` and `torch.manual_seed` fix the random number generators, so subsets and results can be reproduced.
- `torch.device("cuda" if torch.cuda.is_available() else "cpu")` selects the GPU when one exists. Both the model and its input tensors must be placed on this device.

**B.2 Tokenizer and model**
- `AutoTokenizer.from_pretrained("t5-small")` and `AutoModelForSeq2SeqLM.from_pretrained("t5-small")`:
  - The `Auto*` classes read the checkpoint's `config.json` and create the matching classes (`T5TokenizerFast`, `T5ForConditionalGeneration`).
  - They download the vocabulary and weights once and then reuse the local cache.
  - `Seq2SeqLM` stands for an encoder–decoder model with a language-modelling head on the decoder, which is what translation needs.
- `model.to(device)` moves the weights to the GPU or CPU.
- `model.eval()` turns off dropout, so inference is deterministic.
- The tokenizer demo shows SentencePiece pieces. `▁` marks the start of a word, and the ID sequence ends with `</s>` (ID 1).

**B.3 Dataset**
- `load_dataset("wmt/wmt16", "ro-en")` returns a `DatasetDict` with `train` (≈610k pairs), `validation` (1,999) and `test` (1,999) splits. Each record is `{"translation": {"en": ..., "ro": ...}}`.
- We only need the **test** split because we evaluate a pre-trained model and do not train it.
- `.shuffle(seed).select(range(n))` takes a reproducible random subset for CPU runs.
- List comprehensions separate the English sources from the Romanian references.

**B.4 Translate method**
1. **Prefix.** Every sentence starts with `"translate English to Romanian: "`. T5 is multi-task, and the prefix tells it which task to perform.
2. **Batching.** The sentences are processed in batches of 32, which makes good use of the GPU while keeping memory use under control.
3. **Tokenizing.** `tokenizer(..., padding=True, truncation=True, return_tensors="pt")` converts each batch into `input_ids` and an `attention_mask`, both PyTorch tensors:
   - Padding makes all rows the same length.
   - The attention mask tells the encoder to ignore the pad tokens.
   - Truncation protects against very long inputs.
4. **Generating.** `model.generate(**inputs, num_beams=4, max_length=256, early_stopping=True)` runs the encoder once and then decodes step by step:
   - Beam search keeps the 4 best partial hypotheses.
   - `early_stopping` ends the search once all beams have produced `</s>`.
5. **No gradients.** `torch.no_grad()` turns off gradient tracking, which saves memory and time.
6. **Decoding.** `tokenizer.batch_decode(..., skip_special_tokens=True)` maps the IDs back to strings and removes `<pad>` and `</s>`.

**B.5 BLEU**
- `evaluate.load("sacrebleu")` wraps SacreBLEU. It applies its own standard tokenization (`13a`), so we pass it untokenized text.
- References are wrapped as `[[r] for r in references]` because BLEU supports several references per sentence. WMT16 provides one.
- The returned values are:
  - `score`: corpus BLEU on a 0–100 scale.
  - `precisions`: the 1- to 4-gram precisions.
  - `bp`: the brevity penalty.
  - `sys_len` and `ref_len`: the total lengths of the outputs and of the references.
- The extra cells:
  - normalize the Romanian diacritics (the ş/ș issue) and score again;
  - compute sentence-level BLEU to find the best and worst outputs;
  - group BLEU by sentence length;
  - compare greedy decoding with beam search.

## C.2 Explaining the evaluation results (B.5)

*The exact numbers are printed by the cells above. Re-run them after changing the model or the subset. The interpretation below is what to look for.*

1. **Overall BLEU.**
   - Raffel et al. (2020, Table 14) report roughly **27 BLEU** for `t5-small` on WMT English→Romanian, so a score in the **mid-20s** is expected.
   - Small differences come from decoding settings, the maximum length, the diacritics handling and the SacreBLEU version.
   - A score in the 20–30 range usually means translations that a Romanian reader understands and that are mostly correct, but that still contain errors in word choice, agreement or word order.
   - For context, the best WMT16 EN→RO systems scored about 28–29 BLEU (Bojar et al., 2016). A 60M-parameter model used without fine-tuning is therefore competitive with the 2016 state of the art.
2. **n-gram precisions.**
   - Unigram precision is much higher (often ≈ 55–60%) than 4-gram precision (often ≈ 15–20%).
   - The model usually picks the right words but gets exact multi-word sequences right less often.
   - Romanian has rich inflection: case, gender, enclitic articles (*casa* = "the house") and verb conjugation. A single wrong suffix breaks every n-gram that contains the word, so higher-order precisions fall quickly.
3. **Brevity penalty and length ratio.**
   - A length ratio close to 1.0 gives BP ≈ 1, so the score reflects precision only.
   - If the ratio is below 1, the model is dropping content (often clauses at the end of long sentences), and BP lowers the score.
4. **Diacritics.**
   - If the normalized score is noticeably higher than the raw one, part of the "error" was only the cedilla/comma-below encoding mismatch.
   - Reporting both scores makes the comparison fair.
5. **Error analysis.**
   - The worst sentences are typically long, contain named entities, numbers or idioms, or have a reference that paraphrases freely.
   - In the free-paraphrase case a sound translation still scores low, which is a known weakness of BLEU.
   - The length-bucket chart usually shows BLEU falling for long sentences.
6. **Decoding.** Beam search normally beats greedy decoding by about 0.5–1.5 BLEU, because it searches more of the output space.
7. **Overall.**
   - `t5-small` produces usable English→Romanian translations without any fine-tuning.
   - Quality is limited by model size (≈60M parameters), by domain mismatch with the WMT16 news test set, and by the single-reference, surface-overlap nature of BLEU.
   - Ways to improve:
     - use a larger model (`t5-base` or `t5-large`, or the dedicated `Helsinki-NLP/opus-mt-en-ro`);
     - fine-tune on the WMT16 training split;
     - add metrics such as chrF or COMET, which handle morphology and meaning better.

---
# Extension: Romanian → English (Foreign Language → English)

The lab title refers to translation *into* English. T5 was only trained in the English→foreign direction, so for Romanian→English we use `Helsinki-NLP/opus-mt-roa-en`, a MarianMT model trained on OPUS data that translates Romance languages (including Romanian) into English (Tiedemann & Thottingal, 2020). The tokenizer, `generate` and BLEU code stay the same. Only the checkpoint and the direction of the data change, and Marian models do not need a task prefix.

In [ ]:
# Note: there is no "Helsinki-NLP/opus-mt-ro-en" model on the Hub (that name gives an OSError).
# Romanian -> English is covered by the Romance-languages -> English model "opus-mt-roa-en".
# If it cannot be downloaded, we fall back to the general many-languages -> English model.
candidates = ["Helsinki-NLP/opus-mt-roa-en", "Helsinki-NLP/opus-mt-mul-en"]
for ro_en_name in candidates:
    try:
        tok_ro_en = AutoTokenizer.from_pretrained(ro_en_name)
        model_ro_en = AutoModelForSeq2SeqLM.from_pretrained(ro_en_name).to(device).eval()
        print("Loaded", ro_en_name)
        break
    except OSError as err:
        print(f"Could not load {ro_en_name}: {err.__class__.__name__}")
else:
    raise RuntimeError("No Romanian->English model could be loaded.")

def translate_ro_en(sentences, batch_size=32, num_beams=4, max_length=256):
    out = []
    for start in tqdm(range(0, len(sentences), batch_size), desc="RO->EN"):
        inputs = tok_ro_en(sentences[start:start + batch_size], return_tensors="pt",
                           padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            ids = model_ro_en.generate(**inputs, num_beams=num_beams, max_length=max_length)
        out.extend(tok_ro_en.batch_decode(ids, skip_special_tokens=True))
    return out

# Source = Romanian references, target = original English sentences
en_predictions = translate_ro_en(references)
res_ro_en = bleu.compute(predictions=en_predictions, references=[[s] for s in sources])
print(f"RO->EN BLEU: {res_ro_en['score']:.2f}")
for i in range(3):
    print(f"\nRO  : {references[i]}\nREF : {sources[i]}\nPRED: {en_predictions[i]}")

---
# References

- Bojar, O., et al. (2016). Findings of the 2016 Conference on Machine Translation (WMT16). *Proceedings of the First Conference on Machine Translation*, 131–198.
- Hugging Face. (2024). *Transformers documentation: Text generation and `AutoModelForSeq2SeqLM`.* https://huggingface.co/docs/transformers
- Hugging Face. (2024). *Datasets documentation* and *Evaluate: SacreBLEU metric card.* https://huggingface.co/docs/datasets ; https://huggingface.co/spaces/evaluate-metric/sacrebleu
- Kudo, T., & Richardson, J. (2018). SentencePiece: A simple and language independent subword tokenizer and detokenizer for neural text processing. *EMNLP: System Demonstrations*, 66–71.
- Papineni, K., Roukos, S., Ward, T., & Zhu, W.-J. (2002). BLEU: A method for automatic evaluation of machine translation. *Proceedings of ACL*, 311–318.
- Post, M. (2018). A call for clarity in reporting BLEU scores. *Proceedings of the Third Conference on Machine Translation*, 186–191.
- Raffel, C., et al. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *Journal of Machine Learning Research, 21*(140), 1–67.
- Surdeanu, M., & Valenzuela-Escárcega, M. A. (2024). *Deep Learning for Natural Language Processing: A Gentle Introduction.* Cambridge University Press. Notebook: https://github.com/clulab/gentlenlp/blob/main/notebooks/chap15_translation_en_to_ro.ipynb
- Tiedemann, J., & Thottingal, S. (2020). OPUS-MT: Building open translation services for the world. *Proceedings of EAMT*, 479–480.
- Vaswani, A., et al. (2017). Attention is all you need. *Advances in Neural Information Processing Systems, 30*.